# milestone 3 - rag pipeline

In [1]:
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 108.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

data = pd.read_csv('train.csv')

knowledge = []
for row_idx in range(data.shape[0]):
    this_row = data.loc[row_idx]
    letter = this_row['answer']
    knowledge.append(str(this_row[letter]))

print("total docs in kb:", len(knowledge))

enc_model = SentenceTransformer('all-MiniLM-L6-v2')
vectors = enc_model.encode(knowledge, show_progress_bar=False)

vec_dim = vectors.shape[1]
search_index = faiss.IndexFlatL2(vec_dim)
search_index.add(vectors)
print("vectors stored:", search_index.ntotal)

total docs in kb: 2000


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

vectors stored: 2000


q1

In [3]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

sample_row = data.loc[150]
question_txt = str(sample_row['prompt'])
choice_letters = ['A', 'B', 'C', 'D', 'E']
choice_texts = [str(sample_row[ch]) for ch in choice_letters]
gt_letter = sample_row['answer']
gt_text = str(sample_row[gt_letter])

pred1 = zsc(question_txt, candidate_labels=choice_texts)
print(pred1)

ans1 = 0
n = len(pred1['labels'])
for k in range(n):
    if pred1['labels'][k] == gt_text:
        ans1 = pred1['scores'][k]

print(round(ans1, 3))

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

{'sequence': 'Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."', 'The butterfly effect is the phenomenon that a small change in the initial conditio

q2

In [4]:
q_vec = enc_model.encode([question_txt])
dists, hits_idx = search_index.search(q_vec, 10)
retrieved_list = list(hits_idx[0])
print(retrieved_list)

found_rank = 0
r = 1
while r <= 10:
    if retrieved_list[r-1] == 150:
        found_rank = r
        break
    r += 1

print(found_rank)

[np.int64(663), np.int64(1701), np.int64(1269), np.int64(1532), np.int64(576), np.int64(847), np.int64(1693), np.int64(1906), np.int64(168), np.int64(150)]
10


q3

In [5]:
ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

top10_docs = []
for did in retrieved_list:
    top10_docs.append(knowledge[did])

pair_list = []
for d in top10_docs:
    pair_list.append([question_txt, d])

scores_ce = ce.predict(pair_list)

paired = []
for k in range(len(retrieved_list)):
    paired.append((retrieved_list[k], scores_ce[k]))

paired_sorted = sorted(paired, key=lambda t: t[1], reverse=True)

rank3 = 0
for k in range(len(paired_sorted)):
    if paired_sorted[k][0] == 150:
        rank3 = k + 1

print(rank3)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

1


q4

In [6]:
row42 = data.loc[42]
p42 = str(row42['prompt'])

v42 = enc_model.encode([p42])
_, top5idx = search_index.search(v42, 5)

pieces = []
for tid in top5idx[0]:
    pieces.append(knowledge[tid])

joined = " ".join(pieces)
full_str = "Context: " + joined + " Question: " + p42

tokz = AutoTokenizer.from_pretrained('bert-base-uncased')
result4 = tokz(full_str, truncation=False)
token_count = len(result4['input_ids'])
print(token_count)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

216


q5

In [7]:
actual_doc = knowledge[150]
rag_text = "Context: " + actual_doc + " Question: " + question_txt

pred5 = zsc(rag_text, candidate_labels=choice_texts)

ans5 = 0
for k in range(len(pred5['labels'])):
    if pred5['labels'][k] == gt_text:
        ans5 = pred5['scores'][k]

print(round(ans5, 3))

0.989


q6

In [8]:
fake_doc = knowledge[999]
adv_text = "Context: " + fake_doc + " Question: " + question_txt

pred6 = zsc(adv_text, candidate_labels=choice_texts)

ans6 = 0
for k in range(len(pred6['labels'])):
    if pred6['labels'][k] == gt_text:
        ans6 = pred6['scores'][k]

print(round(ans6, 3))

0.529


q7

In [9]:
hit_total = 0

for row_num in range(100):
    r = data.loc[row_num]
    p = str(r['prompt'])
    v = enc_model.encode([p])
    _, ids5 = search_index.search(v, 5)

    docs5 = [knowledge[i] for i in ids5[0]]
    gt = str(r[r['answer']])

    match = False
    for d in docs5:
        if gt in d:
            match = True

    if match == True:
        hit_total = hit_total + 1

pct = (hit_total / 100) * 100
print(round(pct, 1))

73.0


q8

In [10]:
def compute_map3(true_val, top3_vals):
    score = 0
    for pos in range(len(top3_vals)):
        if top3_vals[pos] == true_val:
            score = 1 / (pos + 1)
            break
    return score

scores_list = []

for row_num in range(20):
    r = data.loc[row_num]
    p = str(r['prompt'])

    v = enc_model.encode([p])
    _, ids5 = search_index.search(v, 5)
    docs5 = [knowledge[i] for i in ids5[0]]

    pairs5 = [[p, d] for d in docs5]
    ce_out = ce.predict(pairs5)
    top_pick = docs5[np.argmax(ce_out)]

    rag_str = "Context: " + top_pick + " Question: " + p
    row_choices = [str(r[ch]) for ch in choice_letters]

    pred = zsc(rag_str, candidate_labels=row_choices)

    zipped = list(zip(pred['labels'], pred['scores']))
    zipped.sort(key=lambda pair: pair[1], reverse=True)

    top3 = []
    for idx in range(3):
        txt = zipped[idx][0]
        for ch in choice_letters:
            if str(r[ch]) == txt:
                top3.append(ch)

    true_letter = r['answer']
    s = compute_map3(true_letter, top3)
    scores_list.append(s)

final_val = sum(scores_list) / len(scores_list)
print(round(final_val, 3))

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


0.975
